# Cоздание скриптов для сбора погоды по городам России и визуализация погодных данных

# Описание проекта

В нашем распоряжении данные погодные данные для городов РФ на сайте https://open-meteo.com.

# Цель проекта

Создание погодного скрипта, который собирает погодные данные для городов РФ с сайта https://open-meteo.com, обрабатывает их и сохраняет в базу данных. На основании полученных погодных данных построить визуализацию в Yandex DataLens.


# Задачи проекта

Этап 1: Подготовка и сбор геоданных

    1.1 Реализовать парсер данных Википедии
    - Разработать скрипт для автоматического сбора актуального списка городов России
    - Извлечь информацию о населении, регионах и федеральных округах
    1.2 Интегрировать сервис геокодирования Nominatim
    - Настроить API для определения географических координат (широта/долгота)
    - Реализовать обработку особых случаев (Крым, спорные территории)
    1.3 Подключить OpenTopoData API
    - Реализовать получение высот над уровнем моря по координатам
    - Настроить обработку ошибок и повторные запросы
    1.4 Разработать модуль работы с SQLite
    - Спроектировать структуру базы данных для хранения геоданных
    - Реализовать пакетную обработку и инкрементальное обновление 
Этап 2: Сбор и обработка метеоданных

    2.1 Интегрировать Open-Meteo API
    - Настроить получение исторических погодных данных за период с 01.01.1975 по 31.12.2024 г.
    - Реализовать почасовой сбор метеорологической информации с шагом 6 часов
    2.2 Разработать систему сбора комплексных метеопараметров
    - Реализовать сбор 12+ погодных показателей (температура, влажность, осадки, ветер, снежный покров)
    - Настроить валидацию и очистку получаемых данных
    2.3 Создать механизм обработки данных по городам
    - Реализовать сопоставление метеоданных с географическими координатами
    - Настроить систему очередности и приоритетов обработки

Этап 3: Миграция и развертывание

    3.1 Выполнить миграцию данных в PostgreSQL
    - Разработать скрипты переноса данных из SQLite в PostgreSQL
    - Обеспечить целостность и сохранность данных при миграции
    3.1 Развернуть и настроить сервер PostgreSQL в WSL
    - Установить и настроить PostgreSQL в Ubuntu WSL
    - Настроить статический IP-адрес и проброс портов
    3.2 Обеспечить защиту БД от кибератак
    - Настроить аутентификацию и авторизацию
    - Реализовать базовые меры безопасности (firewall, SSL)
    - Настроить регулярное резервное копирование
    3.3 Настроить сетевое подключение
    - Протестировать подключение к БД из внешней сети
    - Настроить безопасный удаленный доступ

Этап 4: Визуализация и анализ

    4.1 Создать дашборд в Yandex DataLens
    - Разработать структуру дашборда для визуализации данных
    - Настроить подключение к PostgreSQL
    - Реализовать основные графики и отчеты
    4.2 Протестировать работу системы
    - Провести комплексное тестирование всех компонентов
    - Оптимизировать производительность запросов
    - Подготовить документацию по проекту

# Содержание проекта

## Импорт и установка необходимых библиотек

### Устанавливаем библиотеку openmeteo_requests

In [1]:
pip -q install openmeteo_requests

Note: you may need to restart the kernel to use updated packages.


### Устанавливаем библиотеку requests_cache

In [2]:
pip -q install requests_cache

Note: you may need to restart the kernel to use updated packages.


### Устанавливаем библиотеку retry_requests

In [3]:
pip -q install retry_requests

Note: you may need to restart the kernel to use updated packages.


### Устанавливаем библиотеку geopy

In [4]:
!pip -q install geopy

### Импортируем необходимые библиотеки

In [5]:
# =========================
# Стандартные библиотеки
# =========================

import logging
import os
import re
import shutil
import sqlite3
import time
import traceback

from datetime import datetime
from pathlib import Path
from urllib.parse import quote_plus

# =========================
# Сторонние библиотеки
# =========================

import openmeteo_requests
import pandas as pd
import psutil
import psycopg2
import requests
import requests_cache
import sqlalchemy

from bs4 import BeautifulSoup
from geopy.geocoders import Nominatim
from retry_requests import retry
from sqlalchemy import create_engine, text
from tqdm.notebook import tqdm

# =========================
# Специфично для Google Colab
# =========================

from google.colab import drive

## Скрипт для получения городов

### Настраиваем логирование

In [ ]:
logger = logging.getLogger('CityDataCollector')
logger.setLevel(logging.INFO)
if not logger.hasHandlers():
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

    # Логирование в файл
    file_handler = logging.FileHandler('city_processing.log')
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    # Логирование в консоль
    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

### Прописываем константы

In [ ]:
DB_PATH = 'all_russian_cities.db'
WIKI_URL = 'https://ru.wikipedia.org/wiki/Список_городов_России'
BATCH_SIZE = 50  # Обрабатывать по 50 городов за запуск

### Инициализируем геокодер с увеличенным таймаутом

In [ ]:
geolocator = Nominatim(user_agent="city_data_collector/1.0", timeout=10)

### Создадим функцию create_database_table() для создания таблицы в базе данных для хранения информации о городах

Структура таблицы:
- id - уникальный идентификатор (автоинкремент)
- city - название города (обязательное поле)
- region - регион (обязательное поле)
- federal_district -  федеральный округ
- population - численность населения
- latitude - широта
- longitude - долгота
- elevation - высота над уровнем моря
- processed - флаг, отмечающий, был ли город полностью обработан

In [ ]:
def create_database_table(conn):
    conn.execute("""
    CREATE TABLE IF NOT EXISTS cities (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        city TEXT NOT NULL,
        region TEXT NOT NULL,
        federal_district TEXT,
        population INTEGER,
        latitude REAL,
        longitude REAL,
        elevation REAL,
        processed BOOLEAN DEFAULT 0
    );
    """)
    conn.commit()

### Создадим функцию get_wikipedia_cities() для получения актуального списка всех городов России из Википедии

In [ ]:
def get_wikipedia_cities():
    def clean_city_name(name):
        name = str(name)
        name = re.sub(r'\(.*?\)', '', name)  # Удаляем скобки
        name = re.sub(r'(не призн\.?|спорн\.?|временно.*)', '', name, flags=re.IGNORECASE)
        return name.strip()

    try:
        response = requests.get(WIKI_URL, timeout=10)
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'html.parser')

        # Ищем таблицу с городами
        for table in pd.read_html(str(soup)):
            if {'Город', 'Регион', 'Федеральный округ', 'Население'}.issubset(table.columns):
                cleaned_table = table.rename(columns={
                    'Город': 'city',
                    'Регион': 'region',
                    'Федеральный округ': 'federal_district',
                    'Население': 'population'
                }).dropna(subset=['city'])  # Убрали .drop_duplicates()

                cleaned_table['city'] = cleaned_table['city'].apply(clean_city_name)
                return cleaned_table

        raise ValueError("Не найдена таблица с городами на странице Википедии")
    except Exception as e:
        logger.error(f"Ошибка при получении данных из Википедии: {e}")
        raise

### Создадим функцию get_elevation() для получения высоты над уровнем моря через OpenTopoData API

In [ ]:
def get_elevation(lat, lon):
    try:
        url = f"https://api.opentopodata.org/v1/aster30m?locations={lat},{lon}"
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        data = r.json()
        if data.get("results") and "elevation" in data["results"][0]:
            return data["results"][0]["elevation"]
    except Exception as e:
        logger.warning(f"Ошибка получения высоты для ({lat},{lon}): {e}")
    return None

### Создадим функцию geocode_city() для определения координат города с использованием геокодера

In [ ]:
def geocode_city(city_name, region=None, retries=2):
    queries = []

    # Особый случай для крымских городов
    if region and "Крым" in str(region):
        queries.extend([
            f"{city_name}, Крым, Россия",
            f"{city_name}, Крым",
            f"{city_name}, Россия",  # Без указания Крыма
            city_name  # Только название города
        ])
    else:
        queries.extend([
            f"{city_name}, {region}, Россия",
            f"{city_name}, Россия",
            city_name
        ] if region else [f"{city_name}, Россия", city_name])

    for attempt in range(retries):
        for query in queries:
            try:
                location = geolocator.geocode(query, exactly_one=True, timeout=10)
                if location:
                    return location.latitude, location.longitude, region
                sleep(1)
            except Exception as e:
                logger.warning(f"Попытка {attempt+1} для '{query}': {e}")
                sleep(2)

    logger.error(f"Не удалось определить координаты для: {city_name}")
    return None, None, None

### Создадим функцию save_to_database() сохранения данных о городе в базу данных

In [ ]:
def save_to_database(conn, city_data):
    try:
        conn.execute("""
            INSERT INTO cities
            (city, region, federal_district, population, latitude, longitude, elevation, processed)
            VALUES (?, ?, ?, ?, ?, ?, ?, 1)
        """, (
            city_data['city'],
            city_data['region'],
            city_data.get('federal_district'),
            city_data.get('population'),
            city_data['latitude'],
            city_data['longitude'],
            city_data['elevation']
        ))
        conn.commit()
        return True
    except Exception as e:
        logger.error(f"Ошибка при сохранении {city_data['city']}, {city_data['region']}: {e}")
        return False

### Создадим функцию process_cities(), которая будет загружать данные из Википедии, фильтровать по пользовательскому списку, определять координаты и сохранять в базу данных

In [ ]:
def process_cities():
    logger.info("=== Начало обработки данных о городах ===")
    try:
        # Получаем данные из Википедии
        logger.info("Получаем список городов из Википедии...")
        wiki_cities = get_wikipedia_cities()
        logger.info(f"Загружено {len(wiki_cities)} городов из Википедии")

        if wiki_cities.empty:
            logger.warning("Нет городов для обработки!")
            return

        # Подключаемся к базе данных
        with sqlite3.connect(DB_PATH) as conn:
            create_database_table(conn)

            # Получаем необработанные города
            processed = pd.read_sql("SELECT city FROM cities WHERE processed = 1", conn)
            unprocessed = wiki_cities[~wiki_cities['city'].isin(processed['city'])]
            cities_to_process = unprocessed.head(BATCH_SIZE)

            logger.info(f"Городов для обработки в этом запуске: {len(cities_to_process)}")
            if cities_to_process.empty:
                logger.info("Все города уже обработаны.")
                return

            added = updated = skipped = 0

            # Обрабатываем каждый город
            for _, row in cities_to_process.iterrows():
                city = row['city']
                region = row.get('region')

                logger.info(f"Обработка: {city}, {region}")

                # Определяем координаты
                lat, lon, detected_region = geocode_city(city, region)

                if lat and lon:
                    elevation = get_elevation(lat, lon)
                    city_data = {
                        'city': city,
                        'region': detected_region or region,
                        'federal_district': row.get('federal_district'),
                        'population': row.get('population'),
                        'latitude': lat,
                        'longitude': lon,
                        'elevation': elevation
                    }

                    if save_to_database(conn, city_data):
                        added += 1
                        logger.info(f"Добавлен/обновлен: {city}")
                else:
                    skipped += 1
                    logger.warning(f"Пропущен: {city}")

                sleep(1.5)  # Задержка для API

            # Итоговая статистика
            total_in_db = conn.execute("SELECT COUNT(*) FROM cities").fetchone()[0]
            processed_count = conn.execute("SELECT COUNT(*) FROM cities WHERE processed = 1").fetchone()[0]
            logger.info(
                f"Обработка завершена. Добавлено/обновлено: {added}, "
                f"Пропущено: {skipped}"
            )
            logger.info(f"Всего городов в базе: {total_in_db}")
            logger.info(f"Обработано городов: {processed_count}")
            logger.info(f"Осталось обработать: {len(wiki_cities) - processed_count}")

    except Exception as e:
        logger.error(f"Критическая ошибка: {e}", exc_info=True)
    finally:
        logger.info("=== Обработка завершена ===")

### Применим функцию process_cities() для создания базы данных с городами

In [ ]:
process_cities()

## Скрипт для сбора погоды по городам

### Подключаемся к гугл диску

In [ ]:
drive.mount('/content/drive')

### Прописываем константы

In [ ]:
DB_PATH = '/content/drive/MyDrive/Colab Notebooks/Проект скрипта для сбора погоды/all_russian_cities.db'
LOG_FILE = '/content/drive/MyDrive/Colab Notebooks/Проект скрипта для сбора погоды/weather_collector.log'
START_YEAR = 1975
END_YEAR = 2024
REQUEST_DELAY = 15  # задержка между запросами
MAX_ERRORS = 3      # максимальное число подряд ошибок перед остановкой

### Настраиваем логирование

In [ ]:
def setup_logger(LOG_FILE: str, log_level=logging.INFO):
    """
    Настраивает логгер с выводом в файл и консоль. Повторные вызовы безопасны.
    """
    log_dir = Path().parent
    log_dir.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger()
    logger.setLevel(log_level)

    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

    if not any(isinstance(h, logging.FileHandler) and h.baseFilename == str(Path(LOG_FILE).resolve()) for h in logger.handlers):
        file_handler = logging.FileHandler(LOG_FILE, mode='a', encoding='utf-8')
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)

    if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
        console_handler = logging.StreamHandler()
        console_handler.setFormatter(formatter)
        logger.addHandler(console_handler)

    return logger

### Создаем клиент Open-Meteo

In [ ]:
cache_session = requests_cache.CachedSession('.cache', expire_after=86400, allowable_methods=['GET', 'POST'])
retry_session = retry(cache_session, retries=3, backoff_factor=0.5)
openmeteo = openmeteo_requests.Client(session=retry_session)

### Создадим функцию prepare_connection() для оптимизации запросов к базе данных

In [ ]:
def prepare_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("PRAGMA cache_size=-10000")
    return conn

### Создадим функцию create_tables() для создания таблицы в базе данных для хранения информации о городах, погоде в них и отслеживания прогресса в них

Структура таблицы weather (погодные данные):
- date - дата и время измерения погодных данных (в формате YYYY-MM-DD HH:MM:SS)
- city - название города, для которого записаны данные
- latitude - географическая широта местности (в градусах)
- longitude - географическая долгота местности (в градусах)
- temperature_2m - температура воздуха на высоте 2 метров (°C)
- relative_humidity_2m - относительная влажность на высоте 2 метров (%)
- rain - количество дождя (мм)
- snowfall - количество выпавшего снега (см)
- snow_depth - высота снежного покрова (м)
- is_day - флаг дня/ночи (1 — день, 0 — ночь)
- precipitation - общее количество осадков (мм)
- wind_direction_100m - направление ветра на высоте 100 метров (градусы)
- wind_speed_100m - скорость ветра на высоте 100 метров (м/с)

Структура таблицы cities (список городов):
- city - название города
- latitude - географическая широта города
- longitude - географическая долгота города

Структура таблицы progress (прогресс сбора данных):

- city - название города
- latitude - широта города
- longitude - долгота города
- year - год, за который собираются данные
- status - статус сбора данных:
'pending' — ожидает обработки,
'completed' — успешно собраны,
'failed' - ошибка при сборе.
- attempts - количество попыток сбора данных (по умолчанию 0)
- last_attempt - время последней попытки

In [ ]:
def create_tables():
    with sqlite3.connect(DB_PATH) as conn:
        conn.executescript("""
        CREATE TABLE IF NOT EXISTS weather (
            date TEXT,
            city TEXT,
            latitude REAL,
            longitude REAL,
            temperature_2m REAL,
            relative_humidity_2m REAL,
            rain REAL,
            snowfall REAL,
            snow_depth REAL,
            is_day INTEGER,
            precipitation REAL,
            wind_direction_100m REAL,
            wind_speed_100m REAL,
            PRIMARY KEY (date, city, latitude, longitude)
        );

        CREATE TABLE IF NOT EXISTS cities (
            city TEXT,
            latitude REAL,
            longitude REAL,
            PRIMARY KEY (city, latitude, longitude)
        );

        CREATE TABLE IF NOT EXISTS progress (
            city TEXT,
            latitude REAL,
            longitude REAL,
            year INTEGER,
            status TEXT CHECK(status IN ('pending', 'completed', 'failed')),
            attempts INTEGER DEFAULT 0,
            last_attempt TEXT,
            PRIMARY KEY (city, latitude, longitude, year)
        );
        """)

        cities = conn.execute("SELECT city, latitude, longitude FROM cities").fetchall()
        for city, lat, lon in cities:
            for year in range(START_YEAR, END_YEAR + 1):
                conn.execute("""
                INSERT OR IGNORE INTO progress (city, latitude, longitude, year, status)
                VALUES (?, ?, ?, ?, 'pending')
                """, (city, lat, lon, year))
        conn.commit()

### Создадим функцию update_progress() для обновления статуса и информации о попытке обработки для конкретной задачи в таблице progress

In [ ]:
def update_progress(city, lat, lon, year, new_status, increment_attempt_count):
    conn = prepare_connection()  # Используем оптимизированное соединение
    with conn:
        current_time = datetime.now().isoformat()
        if increment_attempt_count:
            conn.execute("""
            UPDATE progress
            SET status = ?, attempts = attempts + 1, last_attempt = ?
            WHERE city = ? AND latitude = ? AND longitude = ? AND year = ?
            """, (new_status, current_time, city, lat, lon, year))
        else:
            conn.execute("""
            UPDATE progress
            SET status = ?, last_attempt = ?
            WHERE city = ? AND latitude = ? AND longitude = ? AND year = ?
            """, (new_status, current_time, city, lat, lon, year))
        conn.commit()

### Создадим функцию process_city() для обработки погодных данных для одного города за определенный диапазон лет с возможностью повторных попыток при ошибках

In [ ]:
def process_city(city, lat, lon, logger=None, year_pbar=None):
    """Обрабатывает все годы для одного города с повторными попытками"""
    if logger is None:
        logger = logging.getLogger()

    consecutive_errors = 0
    years_to_process = list(range(START_YEAR, END_YEAR + 1))

    while years_to_process:
        conn = prepare_connection()  # Используем подготовленное соединение
        with conn:
            cursor = conn.cursor()
            cursor.execute("""
                SELECT year
                FROM progress
                WHERE city = ? AND latitude = ? AND longitude = ?
                  AND status != 'completed'
                  AND attempts < ?
                ORDER BY year
            """, (city, lat, lon, MAX_ERRORS))
            years_to_process = [row[0] for row in cursor.fetchall()]

        if not years_to_process:
            break

        logger.info(f"Обработка города {city} ({lat}, {lon}), годы: {min(years_to_process)}-{max(years_to_process)}")

        try:
            weather_data = fetch_weather_data(lat, lon, years_to_process, logger=logger)

            # === 1. API ЛИМИТ ===
            if weather_data is False or weather_data is None:
                consecutive_errors += 1
                logger.error(f"Лимит API. Попытка {consecutive_errors}/{MAX_ERRORS}")
                for year in years_to_process:
                    update_progress(city, lat, lon, year, 'pending', True)
                    if year_pbar:
                        year_pbar.set_postfix({'год': year, 'статус': 'лимит API'})
                        year_pbar.update(1)
                continue  # Следующая попытка

            # === 2. УСПЕШНО ===
            all_success = True
            for year, year_df in weather_data.items():
                if year_pbar:
                    year_pbar.set_postfix({'год': year, 'статус': 'обработка'})

                if year_df is not None and not year_df.empty:
                    saved = save_weather_data(city, lat, lon, year_df)
                    if saved > 0:
                        update_progress(city, lat, lon, year, 'completed', True)
                        if year_pbar:
                            year_pbar.set_postfix({'год': year, 'статус': 'успешно'})
                    else:
                        logger.warning(f"Данные получены, но не сохранены для {city} ({year})")
                        update_progress(city, lat, lon, year, 'pending', True)
                        all_success = False
                        if year_pbar:
                            year_pbar.set_postfix({'год': year, 'статус': 'ошибка сохранения'})
                else:
                    logger.warning(f"Нет данных для {city} ({year})")
                    update_progress(city, lat, lon, year, 'failed', True)
                    all_success = False
                    if year_pbar:
                        year_pbar.set_postfix({'год': year, 'статус': 'нет данных'})

                if year_pbar:
                    year_pbar.update(1)

            if all_success:
                logger.info(f"Успешно обработан город {city}")
                return True
            else:
                consecutive_errors += 1
                logger.warning(f"Частичный успех для {city}, остались необработанные годы")

        except Exception as e:
            consecutive_errors += 1
            logger.error(f"Ошибка при обработке {city}: {str(e)}")
            for year in years_to_process:
                update_progress(city, lat, lon, year, 'failed', True)
                if year_pbar:
                    year_pbar.set_postfix({'год': year, 'статус': 'ошибка'})
                    year_pbar.update(1)

        if consecutive_errors <= MAX_ERRORS:
            time.sleep(REQUEST_DELAY)

    if consecutive_errors >= MAX_ERRORS:
        logger.error(f"Достигнут лимит попыток ({MAX_ERRORS}) для города {city}")
    return False

### Создадим функцию fetch_weather_data() для запроса погодных данных с Open-Meteo API по координатам и году

In [ ]:
def fetch_weather_data(lat, lon, years, logger=None):
    """
    Запрашивает погодные данные с Open-Meteo API по координатам и диапазону годов.
    Возвращает:
    - словарь {год: DataFrame} при успехе
    - False при достижении лимита API
    - None при других ошибках
    """
    if logger is None:
        logger = logging.getLogger()

    start_year = min(years)
    end_year = max(years)
    start_date = f"{start_year}-01-01"
    end_date = f"{end_year}-12-31"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": [
            "temperature_2m", "relative_humidity_2m", "rain", "snowfall",
            "snow_depth", "is_day", "precipitation", "wind_direction_100m",
            "wind_speed_100m"
        ],
        "windspeed_unit": "ms",
        "temporal_resolution": "hourly_6",
        "timezone": "auto"
    }

    try:
        time.sleep(REQUEST_DELAY * 0.5)  # важная задержка

        responses = openmeteo.weather_api(
            "https://archive-api.open-meteo.com/v1/archive",
            params=params
        )
        response = responses[0]

        # Проверка на ошибки API
        if hasattr(response, 'error') and response.error:
            error_reason = getattr(response, 'reason', 'Unknown error')
            if "limit exceeded" in error_reason.lower():
                logger.error(f"API Limit: {error_reason}")
                return False
            raise Exception(f"API Error: {error_reason}")

        hourly = response.Hourly()

        # Создание общего DataFrame
        full_df = pd.DataFrame({
            "date": pd.date_range(
                start=pd.to_datetime(hourly.Time(), unit="s", utc=False),
                end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=False),
                freq=pd.Timedelta(seconds=hourly.Interval()),
                inclusive="left"
            ),
            "temperature_2m": list(map(float, hourly.Variables(0).ValuesAsNumpy())),
            "relative_humidity_2m": list(map(float, hourly.Variables(1).ValuesAsNumpy())),
            "rain": list(map(float, hourly.Variables(2).ValuesAsNumpy())),
            "snowfall": list(map(float, hourly.Variables(3).ValuesAsNumpy())),
            "snow_depth": list(map(float, hourly.Variables(4).ValuesAsNumpy())),
            "is_day": list(map(float, hourly.Variables(5).ValuesAsNumpy())),
            "precipitation": list(map(float, hourly.Variables(6).ValuesAsNumpy())),
            "wind_direction_100m": list(map(float, hourly.Variables(7).ValuesAsNumpy())),
            "wind_speed_100m": list(map(float, hourly.Variables(8).ValuesAsNumpy())),
        })

        full_df["year"] = full_df["date"].dt.year

        # Разделение по годам
        result = {}
        for year in years:
            year_df = full_df[full_df["year"] == year].copy()
            year_df.drop(columns=["year"], inplace=True)
            if not year_df.empty:
                result[year] = year_df

        return result

    except Exception as e:
        logger.error(f"Ошибка API для {years} ({lat}, {lon}): {str(e)}")
        return None

### Создадим функцию save_weather_data() для сохранения метеорологических данных в базу данных SQLite

In [ ]:
def save_weather_data(city, lat, lon, df):
    if df is None or df.empty:
        return 0
    try:
        df['date'] = df['date'].dt.strftime('%Y-%m-%d %H:%M:%S')
        df['city'] = city
        df['latitude'] = lat
        df['longitude'] = lon

        columns = [
            'date', 'city', 'latitude', 'longitude',
            'temperature_2m', 'relative_humidity_2m',
            'rain', 'snowfall', 'snow_depth', 'is_day',
            'precipitation', 'wind_direction_100m', 'wind_speed_100m'
        ]
        df = df[columns]

        conn = prepare_connection()   # Используем подготовленное соединение
        with conn:
            conn.executemany("""
            INSERT OR IGNORE INTO weather VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?
            )
            """, df.to_records(index=False))
            return len(df)
    except Exception as e:
        logger.error(f"Ошибка при сохранении в БД: {e}")
        return 0

### Создадим функцию main() для автоматической обработки задачи по сбору погоды из API Open-Meteo

In [ ]:
def main():
    try:
        logger = setup_logger(LOG_FILE)
        logger.info("=== СТАРТ СБОРЩИКА ПОГОДЫ ===")

        # Создаем необходимые директории
        db_dir = Path(DB_PATH).parent
        db_dir.mkdir(parents=True, exist_ok=True)

        # Создаем таблицы в базе данных
        create_tables()
        logger.info("Таблицы в базе данных успешно созданы/проверены")

        # Сброс попыток для записей с 'pending' и 'failed' и превышением MAX_ERRORS
        conn = prepare_connection()   # Используем подготовленное соединение
        with conn:
            cursor = conn.cursor()
            cursor.execute("""
                UPDATE progress
                SET attempts = 0
                WHERE status != 'completed'
                AND attempts >= ?
            """, (MAX_ERRORS,))
            reset_count = cursor.rowcount
            conn.commit()

        if reset_count > 0:
            logger.info(f"Сброшено {reset_count} записей со статусом 'pending' и достижением лимита {MAX_ERRORS} попыток")

        total_processed = 0
        api_limit_reached = False

        # Получаем список всех городов для обработки
        conn = prepare_connection()  # Используем подготовленное соединение
        with conn:
            cursor = conn.cursor()
            cursor.execute("""
                SELECT DISTINCT city, latitude, longitude
                FROM cities
                WHERE EXISTS (
                    SELECT 1 FROM progress
                    WHERE progress.city = cities.city
                      AND progress.latitude = cities.latitude
                      AND progress.longitude = cities.longitude
                      AND progress.status != 'completed'
                )
                ORDER BY
                    (
                        SELECT MIN(
                            CASE progress.status
                                WHEN 'failed' THEN 0
                                WHEN 'pending' THEN 1
                                ELSE 2
                            END
                        )
                        FROM progress
                        WHERE progress.city = cities.city
                          AND progress.latitude = cities.latitude
                          AND progress.longitude = cities.longitude
                          AND progress.status != 'completed'
                    ),
                    city
            """)
            cities = cursor.fetchall()

        if not cities:
            logger.info("Нет городов для обработки — все данные уже собраны!")
            return

        # Прогресс-бар по городам
        city_pbar = tqdm(cities, desc="Обработка городов", unit="город")

        for city, lat, lon in city_pbar:
            city_pbar.set_postfix({'город': city[:15] + '...' if len(city) > 15 else city})
            logger.info(f"Начинаем обработку города: {city} ({lat}, {lon})")

            # Получаем список лет для обработки
            conn = prepare_connection()  # Используем подготовленное соединение
            with conn:
                cursor = conn.cursor()
                cursor.execute("""
                    SELECT year
                    FROM progress
                    WHERE city = ? AND latitude = ? AND longitude = ?
                      AND status != 'completed'
                    ORDER BY year
                """, (city, lat, lon))
                years = [row[0] for row in cursor.fetchall()]

            if not years:
                logger.info(f"Нет лет для обработки в городе {city}")
                continue

            # Вложенный прогресс-бар по годам
            year_pbar = tqdm(years, desc="Обработка лет", leave=False, unit="год")

            success = process_city(
                city=city,
                lat=lat,
                lon=lon,
                logger=logger,
                year_pbar=year_pbar
            )

            year_pbar.close()

            if success is False:
                api_limit_reached = True
                break

            if not success:
                logger.error(f"Не удалось обработать город {city} после {MAX_ERRORS} попыток")

            # Подсчёт записей, добавленных в этом запуске
            conn = prepare_connection()  # Используем подготовленное соединение
            with conn:
                cursor = conn.cursor()
                cursor.execute("""
                    SELECT COUNT(*) FROM weather
                    WHERE city = ? AND latitude = ? AND longitude = ?
                """, (city, lat, lon))
                total_processed += cursor.fetchone()[0]

            os.sync()
            time.sleep(REQUEST_DELAY * 0.5)

    except Exception as e:
        logger = logging.getLogger()
        logger.error(f"КРИТИЧЕСКАЯ ОШИБКА: {str(e)}")

    finally:
        # Закрываем прогресс-бары, если они были
        if 'city_pbar' in locals():
            city_pbar.close()
        if 'year_pbar' in locals():
            year_pbar.close()

        if 'api_limit_reached' in locals() and api_limit_reached:
            logger.warning("=== ПРЕРВАНО ИЗ-ЗА ЛИМИТА API ===")

        # Подсчёт общего количества записей в таблице weather
        conn = prepare_connection()  # Используем подготовленное соединение
        with conn:
            cursor = conn.cursor()
            cursor.execute("""SELECT COUNT(*) FROM weather""")
            total_records = cursor.fetchone()[0]

            # Подсчёт успешно обработанных городов (где все годы помечены как completed)
            cursor.execute("""
                SELECT COUNT(*)
                FROM (
                    SELECT city, latitude, longitude
                    FROM progress
                    GROUP BY city, latitude, longitude
                    HAVING MIN(status) = 'completed' AND MAX(status) = 'completed'
               )
            """)
            completed_cities = cursor.fetchone()[0]

        logger.info(f"Всего добавлено записей в таблицу weather в этом запуске: {total_processed}")
        logger.info(f"Всего записей в таблице weather: {total_records}")
        logger.info(f"Всего успешно обработано городов: {completed_cities}")
        logger.info("=== КОНЕЦ СБОРЩИКА ПОГОДЫ ===")

        try:
            gdrive_path = '/content/drive/MyDrive/Colab Notebooks/Проект скрипта для сбора погоды/'

            # Копируем лог
            if os.path.exists(LOG_FILE):
                dest_log_path = os.path.join(gdrive_path, os.path.basename(LOG_FILE))
                if os.path.abspath(LOG_FILE) != os.path.abspath(dest_log_path):
                    shutil.copy(LOG_FILE, dest_log_path)
                    logger.info(f"Лог скопирован в {dest_log_path}")

            # Копируем БД
            if os.path.exists(DB_PATH):
                dest_db_path = os.path.join(gdrive_path, os.path.basename(DB_PATH))
                if os.path.abspath(DB_PATH) != os.path.abspath(dest_db_path):
                    shutil.copy(DB_PATH, dest_db_path)
                    logger.info(f"База данных скопирована в {dest_db_path}")

            # Показываем последние строки логов
            print("\nПоследние записи лога:")
            if os.path.exists(LOG_FILE):
                with open(LOG_FILE, 'r', encoding='utf-8') as f:
                    log_lines = f.readlines()
                    print("".join(log_lines[-20:]))
            else:
                print("Файл логов не найден")

        except Exception as e:
            print(f"Ошибка при сохранении результатов: {e}")
            logger.error(f"Ошибка при копировании файлов: {e}")

### Применим функцию main() для запуска сбора погоды в файл all_russian_cities.db

In [ ]:
main()

## Миграция и развертывание базы данных

### Устранение проблем при миграции базы данных

#### Пропишем путь к базе данных 

In [ ]:
db_path = (r'C:\Users\Равиль\Проекты для портфолио\9. Проект скрипта для сбора погоды\Скрипт для сбора погоды\\' +
          r'all_russian_cities.db')

#### Уберем пробелы в столбце population в таблице cities

In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("""UPDATE cities
                  SET population = REPLACE(population, ' ', '')
                  WHERE population LIKE '% %'""")
conn.commit()
cursor.close()
conn.close()

#### Устраняем проблемные значения в строках 608 и 849 в столбце population в таблице cities

In [ ]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("""UPDATE cities
                  SET population = CASE
                      WHEN id = 608 THEN '13010112'
                      WHEN id = 849 THEN '547820'
                      ELSE population
                  END
                  WHERE id IN (608, 849)""")
conn.commit()
cursor.close()
conn.close()

#### Добавим в таблицу weather столбец id на основе сопоставления координат latitude и longitude каждой записи из таблицы cities

In [ ]:
batch_size = 2000000  # Увеличенный размер пакета

def format_time(seconds):
    """Форматирование времени в читаемый вид"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    sec = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{sec:02d}"

def print_system_stats():
    """Вывод статистики использования ресурсов"""
    mem = psutil.virtual_memory()
    print(f"📊 Использование RAM: {mem.percent}% (доступно: {mem.available/1024/1024:,.0f} MB)")
    print(f"⚡ Использование CPU: {psutil.cpu_percent()}%")

print("⚡ Инициализация обработки 82 миллионов записей")
print(f"🖥️ Конфигурация ПК: AMD Ryzen 7 7700, 64GB RAM, NVMe SSD")
print(f"⚙️ Размер пакета: {batch_size:,} записей".replace(",", " "))
print_system_stats()

start_time = time.time()

try:
    conn = sqlite3.connect(db_path)
    
    # Максимальные оптимизации SQLite
    conn.execute("PRAGMA journal_mode = MEMORY")
    conn.execute("PRAGMA synchronous = OFF")
    conn.execute("PRAGMA cache_size = -1000000")  # Увеличено до 1GB
    conn.execute("PRAGMA temp_store = MEMORY")
    conn.execute("PRAGMA locking_mode = EXCLUSIVE")
    conn.execute("PRAGMA threads = 4")  # Используем больше потоков
    
    print("\n🔍 Анализ данных...")
    total_records = pd.read_sql("SELECT COUNT(*) FROM weather", conn).iloc[0,0]
    print(f"📊 Всего записей в таблице weather: {total_records:,}".replace(",", " "))
    print_system_stats()
    
    # Создание индексов
    print("\n🛠️ Создание индексов для координат...")
    with tqdm(total=2, desc="Создание индексов", ncols=80) as pbar:
        conn.execute("CREATE INDEX IF NOT EXISTS idx_cities_coords ON cities(latitude, longitude)")
        pbar.update(1)
        conn.execute("CREATE INDEX IF NOT EXISTS idx_weather_coords ON weather(latitude, longitude)")
        pbar.update(1)
        conn.commit()
    print_system_stats()
    
    # Добавление столбца id
    print("\n➕ Добавляем столбец id в таблицу weather")
    conn.execute("ALTER TABLE weather ADD COLUMN id INTEGER;")
    conn.commit()
    
    # Оптимизированный алгоритм обработки
    print("\n🚀 Начало оптимизированной обработки данных...")
    
    # 1. Создаем временную таблицу с хэшами координат
    print("🔗 Создание таблицы хэшей координат...")
    conn.execute("DROP TABLE IF EXISTS temp_coord_hashes")
    conn.execute("""
    CREATE TEMP TABLE temp_coord_hashes AS
    SELECT 
        rowid,
        CAST(latitude*10000 AS INTEGER) as lat_hash,
        CAST(longitude*10000 AS INTEGER) as lon_hash
    FROM weather
    """)
    conn.execute("CREATE INDEX idx_temp_hashes ON temp_coord_hashes(lat_hash, lon_hash)")
    conn.commit()
    
    # 2. Быстрое создание таблицы сопоставлений
    print("🔄 Создание таблицы сопоставлений...")
    conn.execute("DROP TABLE IF EXISTS temp_matched_ids")
    conn.execute("""
    CREATE TEMP TABLE temp_matched_ids AS
    SELECT w.rowid, c.id
    FROM temp_coord_hashes w
    JOIN (
        SELECT 
            id,
            CAST(latitude*10000 AS INTEGER) as lat_hash,
            CAST(longitude*10000 AS INTEGER) as lon_hash
        FROM cities
    ) c ON w.lat_hash = c.lat_hash AND w.lon_hash = c.lon_hash
    """)
    conn.commit()
    
    # 3. Создаем индекс для временной таблицы
    print("📊 Индексирование результатов...")
    conn.execute("CREATE INDEX idx_temp_matched ON temp_matched_ids(rowid)")
    conn.commit()
    
    # Пакетное обновление
    print("\n⚡ Пакетное обновление записей...")
    max_rowid = pd.read_sql("SELECT MAX(rowid) FROM weather", conn).iloc[0,0]
    batches = range(1, max_rowid + 1, batch_size)
    
    with tqdm(total=max_rowid, desc="Обновление", unit="запись", 
              ncols=120, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]") as pbar:
        for start in batches:
            conn.execute(f"""
                UPDATE weather
                SET id = (
                    SELECT id FROM temp_matched_ids WHERE rowid = weather.rowid
                )
                WHERE rowid BETWEEN {start} AND {start + batch_size - 1}
            """)
            conn.commit()
            pbar.update(batch_size)
            
            if start % (10 * batch_size) == 1:
                print_system_stats()
    
    # Проверка результатов
    print("\n🔎 Проверка результатов:")
    stats = pd.read_sql("""
        SELECT 
            COUNT(*) as total,
            SUM(CASE WHEN id IS NOT NULL THEN 1 ELSE 0 END) as matched,
            SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) as unmatched
        FROM weather
    """, conn)
    
    print(f"✅ Успешно сопоставлено: {stats['matched'][0]:,} записей".replace(",", " "))
    print(f"⚠️ Не сопоставлено: {stats['unmatched'][0]:,} записей".replace(",", " "))

except Exception as e:
    print(f"\n❌ Ошибка: {str(e)}")
    import traceback
    traceback.print_exc()

finally:
    # Очистка
    if 'conn' in locals():
        print("\n🧹 Очистка временных объектов...")
        conn.execute("DROP TABLE IF EXISTS temp_matched_ids")
        conn.execute("DROP TABLE IF EXISTS temp_coord_hashes")
        conn.commit()
        conn.close()
    print(f"🕒 Общее время выполнения: {format_time(time.time() - start_time)}")
    print("🔌 Соединение с базой данных закрыто")
    print_system_stats()

#### Переместим столбец id таблицы weather в начало, сохраняя все остальные структуры данных

In [ ]:
def format_time(seconds):
    """Форматирование времени в читаемый вид"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    sec = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{sec:02d}"

def get_column_definition(col_def):
    """Извлекает имя и тип столбца, игнорируя ограничения"""
    parts = col_def.strip().split()
    if len(parts) >= 2:
        return f"{parts[0]} {parts[1]}"
    return None

def recreate_index(cursor, original_name, temp_name, table_name):
    """Правильно пересоздает индекс с оригинальным именем"""
    try:
        # Получаем SQL определения временного индекса
        cursor.execute(f"SELECT sql FROM sqlite_master WHERE name = '{temp_name}'")
        index_sql = cursor.fetchone()[0]
        
        # Модифицируем SQL для создания индекса с оригинальным именем
        new_sql = index_sql.replace(temp_name, original_name).replace(f'ON {table_name}', 'ON weather')
        cursor.execute(new_sql)
        
        # Удаляем временный индекс
        cursor.execute(f"DROP INDEX {temp_name}")
        return True
    except Exception as e:
        print(f"Ошибка при восстановлении индекса {original_name}: {str(e)}")
        return False

print("⚡ Точное копирование таблицы с перемещением столбца id")
start_time = time.time()

try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Максимальные оптимизации SQLite
    conn.execute("PRAGMA journal_mode = MEMORY")
    conn.execute("PRAGMA synchronous = OFF")
    conn.execute("PRAGMA cache_size = -2000000")  # 2GB кэша
    conn.execute("PRAGMA temp_store = MEMORY")
    conn.execute("PRAGMA locking_mode = EXCLUSIVE")
    conn.execute("PRAGMA threads = 4")  # Используем больше потоков
    
    # Получаем структуру таблицы
    print("\n🔍 Анализ структуры таблицы weather...")
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='weather'")
    create_script = cursor.fetchone()[0]
    print(f"Оригинальный SQL создания таблицы:\n{create_script}")

    # Извлекаем все определения столбцов
    start_def = create_script.find('(')
    end_def = create_script.rfind(')')
    columns_part = create_script[start_def+1:end_def]
    
    # Обрабатываем каждое определение столбца
    columns = []
    for col_def in columns_part.split(','):
        col_def = col_def.strip()
        if not col_def:
            continue
            
        # Пропускаем ограничения таблицы (PRIMARY KEY и т.д.)
        if col_def.startswith(('PRIMARY KEY', 'UNIQUE', 'CONSTRAINT', 'FOREIGN KEY')):
            continue
            
        col_info = get_column_definition(col_def)
        if col_info:
            columns.append(col_info)

    # Находим столбец id и перемещаем его в начало
    id_columns = [col for col in columns if col.lower().startswith('id ')]
    other_columns = [col for col in columns if not col.lower().startswith('id ')]
    
    if not id_columns:
        raise ValueError("Столбец id не найден в таблице weather")
        
    new_columns = id_columns + other_columns

    # Создаем временную таблицу
    print("🚀 Создание новой таблицы...")
    cursor.execute("BEGIN TRANSACTION")
    temp_table_name = "weather_new_temp"
    cursor.execute(f"DROP TABLE IF EXISTS {temp_table_name}")
    
    # Создаем таблицу с новым порядком столбцов
    create_table_sql = f"""
        CREATE TABLE {temp_table_name} (
            {', '.join(new_columns)}
        )
    """
    print(f"SQL создания временной таблицы:\n{create_table_sql}")
    cursor.execute(create_table_sql)

    # Переносим данные с прогресс-баром
    print("🔄 Перенос данных...")
    col_names = [col.split()[0] for col in new_columns]
    
    # Получаем общее количество строк для прогресс-бара
    cursor.execute("SELECT COUNT(*) FROM weather")
    total_rows = cursor.fetchone()[0]
    
    # Используем tqdm для отображения прогресса
    batch_size = 100000  # Размер пакета для переноса
    with tqdm(total=total_rows, unit='row', desc='Перенос данных') as pbar:
        for offset in range(0, total_rows, batch_size):
            cursor.execute(f"""
                INSERT INTO {temp_table_name}
                SELECT {', '.join(col_names)}
                FROM weather
                LIMIT {batch_size} OFFSET {offset}
            """)
            pbar.update(batch_size)
            conn.commit()  # Периодически фиксируем изменения

    # Восстанавливаем PRIMARY KEY
    if 'PRIMARY KEY' in create_script:
        pk_def = create_script.split('PRIMARY KEY')[1].split(')')[0] + ')'
        temp_pk_name = f"temp_pk_weather"
        print(f"Создаем PRIMARY KEY: {temp_pk_name}")
        cursor.execute(f"""
            CREATE UNIQUE INDEX {temp_pk_name} 
            ON {temp_table_name} {pk_def}
        """)

    # Переносим остальные индексы с временными именами
    print("📊 Перенос индексов...")
    cursor.execute("""
        SELECT name, sql FROM sqlite_master 
        WHERE type='index' AND tbl_name='weather' 
        AND sql IS NOT NULL 
        AND name NOT LIKE 'sqlite_autoindex%'
    """)
    
    index_rename_map = {}
    indexes = cursor.fetchall()
    
    # Прогресс-бар для переноса индексов
    for name, sql in tqdm(indexes, desc='Перенос индексов', unit='index'):
        temp_index_name = f"temp_{name}"
        new_sql = sql.replace('ON weather', f'ON {temp_table_name}')
        new_sql = new_sql.replace(f'INDEX {name}', f'INDEX {temp_index_name}')
        try:
            cursor.execute(new_sql)
            index_rename_map[name] = temp_index_name
        except sqlite3.OperationalError as e:
            print(f"Не удалось создать временный индекс {temp_index_name}: {str(e)}")

    # Заменяем таблицу
    print("🔄 Замена таблиц...")
    cursor.execute("DROP TABLE weather")
    cursor.execute(f"ALTER TABLE {temp_table_name} RENAME TO weather")
    
    # Восстанавливаем индексы с оригинальными именами
    print("🔄 Восстанавливаем имена индексов...")
    for original_name, temp_name in tqdm(index_rename_map.items(), desc='Восстановление индексов', unit='index'):
        if not recreate_index(cursor, original_name, temp_name, temp_table_name):
            print(f"Индекс {original_name} будет отсутствовать! Требуется ручное восстановление.")

    # Особый случай для PRIMARY KEY
    if 'PRIMARY KEY' in create_script:
        pk_def = create_script.split('PRIMARY KEY')[1].split(')')[0] + ')'
        try:
            cursor.execute(f"DROP INDEX temp_pk_weather")
            cursor.execute(f"""
                CREATE UNIQUE INDEX pk_weather 
                ON weather {pk_def}
            """)
        except Exception as e:
            print(f"Ошибка при восстановлении PRIMARY KEY: {str(e)}")
    
    conn.commit()

    # Проверка
    print("\n🔎 Проверка структуры:")
    print(pd.read_sql("PRAGMA table_info(weather)", conn))
    
    # Проверка индексов
    print("\n🔎 Проверка индексов:")
    print(pd.read_sql("SELECT name, tbl_name, sql FROM sqlite_master WHERE type='index' AND tbl_name='weather'", conn))
    
    print(f"\n✅ Готово! Время выполнения: {format_time(time.time() - start_time)}")

except Exception as e:
    print(f"\n❌ Ошибка: {str(e)}")
    conn.rollback()
    import traceback
    traceback.print_exc()

finally:
    if 'conn' in locals():
        conn.close()
    print("\n🔌 Соединение закрыто")

#### Добавим в таблицу progress столбец id на основе сопоставления координат latitude и longitude каждой записи из таблицы cities

In [ ]:
batch_size = 2000000  # Размер пакета для обработки

def format_time(seconds):
    """Форматирование времени в читаемый вид"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    sec = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{sec:02d}"

def print_system_stats():
    """Вывод статистики использования ресурсов"""
    mem = psutil.virtual_memory()
    print(f"📊 Использование RAM: {mem.percent}% (доступно: {mem.available/1024/1024:,.0f} MB)")
    print(f"⚡ Использование CPU: {psutil.cpu_percent()}%")

print("⚡ Инициализация обработки таблицы progress")
print(f"🖥️ Конфигурация ПК: AMD Ryzen 7 7700, 64GB RAM, NVMe SSD")
print(f"⚙️ Размер пакета: {batch_size:,} записей".replace(",", " "))
print_system_stats()

start_time = time.time()

try:
    conn = sqlite3.connect(db_path)
    
    # Максимальные оптимизации SQLite
    conn.execute("PRAGMA journal_mode = MEMORY")
    conn.execute("PRAGMA synchronous = OFF")
    conn.execute("PRAGMA cache_size = -1000000")  # Увеличено до 1GB
    conn.execute("PRAGMA temp_store = MEMORY")
    conn.execute("PRAGMA locking_mode = EXCLUSIVE")
    conn.execute("PRAGMA threads = 4")
    
    print("\n🔍 Анализ данных...")
    total_records = pd.read_sql("SELECT COUNT(*) FROM progress", conn).iloc[0,0]
    print(f"📊 Всего записей в таблице progress: {total_records:,}".replace(",", " "))
    print_system_stats()
    
    # Проверяем, есть ли уже столбец id
    cursor = conn.execute("PRAGMA table_info(progress)")
    columns = [column[1] for column in cursor.fetchall()]
    
    if 'id' not in columns:
        print("\n➕ Добавляем столбец id в таблицу progress")
        conn.execute("ALTER TABLE progress ADD COLUMN id INTEGER;")
        conn.commit()
    else:
        print("\nℹ️ Столбец id уже существует в таблице progress")
        conn.execute("UPDATE progress SET id = NULL")
        conn.commit()
    
    # Создаем индексы
    print("\n🛠️ Создание индексов...")
    with tqdm(total=3, desc="Создание индексов", ncols=80) as pbar:
        conn.execute("CREATE INDEX IF NOT EXISTS idx_cities_coords ON cities(latitude, longitude)")
        pbar.update(1)
        conn.execute("CREATE INDEX IF NOT EXISTS idx_progress_coords ON progress(latitude, longitude)")
        pbar.update(1)
        conn.execute("CREATE INDEX IF NOT EXISTS idx_cities_name ON cities(city)")
        pbar.update(1)
        conn.commit()
    print_system_stats()
    
    # Оптимизированный алгоритм обработки с хэшами координат
    print("\n🚀 Начало обработки данных с хэшами координат...")
    
    # 1. Создаем временную таблицу с хэшами координат для progress
    print("🔗 Создание таблицы хэшей координат для progress...")
    conn.execute("DROP TABLE IF EXISTS temp_progress_coord_hashes")
    conn.execute("""
    CREATE TEMP TABLE temp_progress_coord_hashes AS
    SELECT 
        rowid,
        city,
        CAST(latitude*10000 AS INTEGER) as lat_hash,
        CAST(longitude*10000 AS INTEGER) as lon_hash
    FROM progress
    """)
    conn.execute("CREATE INDEX idx_temp_progress_hashes ON temp_progress_coord_hashes(city, lat_hash, lon_hash)")
    conn.commit()
    
    # 2. Создаем временную таблицу с хэшами координат для cities
    print("🔗 Создание таблицы хэшей координат для cities...")
    conn.execute("DROP TABLE IF EXISTS temp_cities_coord_hashes")
    conn.execute("""
    CREATE TEMP TABLE temp_cities_coord_hashes AS
    SELECT 
        id,
        city,
        CAST(latitude*10000 AS INTEGER) as lat_hash,
        CAST(longitude*10000 AS INTEGER) as lon_hash
    FROM cities
    """)
    conn.execute("CREATE INDEX idx_temp_cities_hashes ON temp_cities_coord_hashes(city, lat_hash, lon_hash)")
    conn.commit()
    
    # 3. Создаем таблицу сопоставлений
    print("🔄 Создание таблицы сопоставлений...")
    conn.execute("DROP TABLE IF EXISTS temp_progress_matched_ids")
    conn.execute("""
    CREATE TEMP TABLE temp_progress_matched_ids AS
    SELECT p.rowid, c.id
    FROM temp_progress_coord_hashes p
    JOIN temp_cities_coord_hashes c 
      ON p.city = c.city 
     AND p.lat_hash = c.lat_hash 
     AND p.lon_hash = c.lon_hash
    """)
    conn.commit()
    
    # 4. Создаем индекс для временной таблицы
    print("📊 Индексирование результатов...")
    conn.execute("CREATE INDEX idx_temp_progress_matched ON temp_progress_matched_ids(rowid)")
    conn.commit()
    
    # Пакетное обновление
    print("\n⚡ Пакетное обновление записей progress...")
    max_rowid = pd.read_sql("SELECT MAX(rowid) FROM progress", conn).iloc[0,0]
    batches = range(1, max_rowid + 1, batch_size)
    
    with tqdm(total=max_rowid, desc="Обновление progress", unit="запись", 
              ncols=120, bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]") as pbar:
        for start in batches:
            conn.execute(f"""
                UPDATE progress
                SET id = (
                    SELECT id FROM temp_progress_matched_ids WHERE rowid = progress.rowid
                )
                WHERE rowid BETWEEN {start} AND {start + batch_size - 1}
            """)
            conn.commit()
            pbar.update(batch_size)
            
            if start % (10 * batch_size) == 1:
                print_system_stats()
    
    # Проверка результатов
    print("\n🔎 Проверка результатов:")
    stats = pd.read_sql("""
        SELECT 
            COUNT(*) as total,
            SUM(CASE WHEN id IS NOT NULL THEN 1 ELSE 0 END) as matched,
            SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) as unmatched,
            (SUM(CASE WHEN id IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) as match_percent
        FROM progress
    """, conn)
    
    print(f"✅ Успешно сопоставлено: {stats['matched'][0]:,} записей ({stats['match_percent'][0]:.2f}%)".replace(",", " "))
    print(f"⚠️ Не сопоставлено: {stats['unmatched'][0]:,} записей".replace(",", " "))
    
    # Дополнительная проверка распределения ID
    print("\n🔢 Проверка распределения ID:")
    id_distribution = pd.read_sql("SELECT id, COUNT(*) as count FROM progress WHERE id IS NOT NULL GROUP BY id ORDER BY count DESC LIMIT 10", conn)
    print(id_distribution)

except Exception as e:
    print(f"\n❌ Ошибка: {str(e)}")
    import traceback
    traceback.print_exc()

finally:
    # Очистка временных таблиц
    if 'conn' in locals():
        print("\n🧹 Очистка временных объектов...")
        conn.execute("DROP TABLE IF EXISTS temp_progress_coord_hashes")
        conn.execute("DROP TABLE IF EXISTS temp_cities_coord_hashes")
        conn.execute("DROP TABLE IF EXISTS temp_progress_matched_ids")
        conn.commit()
        conn.close()
    print(f"🕒 Общее время выполнения: {format_time(time.time() - start_time)}")
    print("🔌 Соединение с базой данных закрыто")
    print_system_stats()

#### Переместим столбец id таблицы progress в начало, сохраняя все остальные структуры данных

In [ ]:
def format_time(seconds):
    """Форматирование времени в читаемый вид"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    sec = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{sec:02d}"

def get_column_definition(col_def):
    """Извлекает имя и тип столбца, игнорируя ограничения"""
    parts = col_def.strip().split()
    if len(parts) >= 2:
        return f"{parts[0]} {parts[1]}"
    return None

def recreate_index(cursor, original_name, temp_name, table_name):
    """Правильно пересоздает индекс с оригинальным именем"""
    try:
        # Получаем SQL определения временного индекса
        cursor.execute(f"SELECT sql FROM sqlite_master WHERE name = '{temp_name}'")
        index_sql = cursor.fetchone()[0]
        
        # Модифицируем SQL для создания индекса с оригинальным именем
        new_sql = index_sql.replace(temp_name, original_name).replace(f'ON {table_name}', 'ON progress')
        cursor.execute(new_sql)
        
        # Удаляем временный индекс
        cursor.execute(f"DROP INDEX {temp_name}")
        return True
    except Exception as e:
        print(f"Ошибка при восстановлении индекса {original_name}: {str(e)}")
        return False

print("⚡ Точное копирование таблицы progress с перемещением столбца id")
start_time = time.time()

try:
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Максимальные оптимизации SQLite
    conn.execute("PRAGMA journal_mode = MEMORY")
    conn.execute("PRAGMA synchronous = OFF")
    conn.execute("PRAGMA cache_size = -2000000")  # 2GB кэша
    conn.execute("PRAGMA temp_store = MEMORY")
    conn.execute("PRAGMA locking_mode = EXCLUSIVE")
    conn.execute("PRAGMA threads = 4")  # Используем больше потоков
    
    # Получаем структуру таблицы
    print("\n🔍 Анализ структуры таблицы progress...")
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='progress'")
    create_script = cursor.fetchone()[0]
    print(f"Оригинальный SQL создания таблицы:\n{create_script}")

    # Извлекаем все определения столбцов
    start_def = create_script.find('(')
    end_def = create_script.rfind(')')
    columns_part = create_script[start_def+1:end_def]
    
    # Обрабатываем каждое определение столбца
    columns = []
    for col_def in columns_part.split(','):
        col_def = col_def.strip()
        if not col_def:
            continue
            
        # Пропускаем ограничения таблицы (PRIMARY KEY и т.д.)
        if col_def.startswith(('PRIMARY KEY', 'UNIQUE', 'CONSTRAINT', 'FOREIGN KEY')):
            continue
            
        col_info = get_column_definition(col_def)
        if col_info:
            columns.append(col_info)

    # Находим столбец id и перемещаем его в начало
    id_columns = [col for col in columns if col.lower().startswith('id ')]
    other_columns = [col for col in columns if not col.lower().startswith('id ')]
    
    if not id_columns:
        raise ValueError("Столбец id не найден в таблице progress")
        
    new_columns = id_columns + other_columns

    # Создаем временную таблицу
    print("🚀 Создание новой таблицы...")
    cursor.execute("BEGIN TRANSACTION")
    temp_table_name = "progress_new_temp"
    cursor.execute(f"DROP TABLE IF EXISTS {temp_table_name}")
    
    # Создаем таблицу с новым порядком столбцов
    create_table_sql = f"""
        CREATE TABLE {temp_table_name} (
            {', '.join(new_columns)}
        )
    """
    print(f"SQL создания временной таблицы:\n{create_table_sql}")
    cursor.execute(create_table_sql)

    # Переносим данные с прогресс-баром
    print("🔄 Перенос данных...")
    col_names = [col.split()[0] for col in new_columns]
    
    # Получаем общее количество строк для прогресс-бара
    cursor.execute("SELECT COUNT(*) FROM progress")
    total_rows = cursor.fetchone()[0]
    
    # Используем tqdm для отображения прогресса
    batch_size = 100000  # Размер пакета для переноса
    with tqdm(total=total_rows, unit='row', desc='Перенос данных') as pbar:
        for offset in range(0, total_rows, batch_size):
            cursor.execute(f"""
                INSERT INTO {temp_table_name}
                SELECT {', '.join(col_names)}
                FROM progress
                LIMIT {batch_size} OFFSET {offset}
            """)
            pbar.update(batch_size)
            conn.commit()  # Периодически фиксируем изменения

    # Восстанавливаем PRIMARY KEY
    if 'PRIMARY KEY' in create_script:
        pk_def = create_script.split('PRIMARY KEY')[1].split(')')[0] + ')'
        temp_pk_name = f"temp_pk_progress"
        print(f"Создаем PRIMARY KEY: {temp_pk_name}")
        cursor.execute(f"""
            CREATE UNIQUE INDEX {temp_pk_name} 
            ON {temp_table_name} {pk_def}
        """)

    # Переносим остальные индексы с временными именами
    print("📊 Перенос индексов...")
    cursor.execute("""
        SELECT name, sql FROM sqlite_master 
        WHERE type='index' AND tbl_name='progress' 
        AND sql IS NOT NULL 
        AND name NOT LIKE 'sqlite_autoindex%'
    """)
    
    index_rename_map = {}
    indexes = cursor.fetchall()
    
    # Прогресс-бар для переноса индексов
    for name, sql in tqdm(indexes, desc='Перенос индексов', unit='index'):
        temp_index_name = f"temp_{name}"
        new_sql = sql.replace('ON progress', f'ON {temp_table_name}')
        new_sql = new_sql.replace(f'INDEX {name}', f'INDEX {temp_index_name}')
        try:
            cursor.execute(new_sql)
            index_rename_map[name] = temp_index_name
        except sqlite3.OperationalError as e:
            print(f"Не удалось создать временный индекс {temp_index_name}: {str(e)}")

    # Заменяем таблицу
    print("🔄 Замена таблиц...")
    cursor.execute("DROP TABLE progress")
    cursor.execute(f"ALTER TABLE {temp_table_name} RENAME TO progress")
    
    # Восстанавливаем индексы с оригинальными именами
    print("🔄 Восстанавливаем имена индексов...")
    for original_name, temp_name in tqdm(index_rename_map.items(), desc='Восстановление индексов', unit='index'):
        if not recreate_index(cursor, original_name, temp_name, temp_table_name):
            print(f"Индекс {original_name} будет отсутствовать! Требуется ручное восстановление.")

    # Особый случай для PRIMARY KEY
    if 'PRIMARY KEY' in create_script:
        pk_def = create_script.split('PRIMARY KEY')[1].split(')')[0] + ')'
        try:
            cursor.execute(f"DROP INDEX temp_pk_progress")
            cursor.execute(f"""
                CREATE UNIQUE INDEX pk_progress 
                ON progress {pk_def}
            """)
        except Exception as e:
            print(f"Ошибка при восстановлении PRIMARY KEY: {str(e)}")
    
    conn.commit()

    # Проверка
    print("\n🔎 Проверка структуры:")
    print(pd.read_sql("PRAGMA table_info(progress)", conn))
    
    # Проверка индексов
    print("\n🔎 Проверка индексов:")
    print(pd.read_sql("SELECT name, tbl_name, sql FROM sqlite_master WHERE type='index' AND tbl_name='progress'", conn))
    
    print(f"\n✅ Готово! Время выполнения: {format_time(time.time() - start_time)}")

except Exception as e:
    print(f"\n❌ Ошибка: {str(e)}")
    conn.rollback()
    import traceback
    traceback.print_exc()

finally:
    if 'conn' in locals():
        conn.close()
    print("\n🔌 Соединение закрыто")

#### Выполним сжатие базы данных для уменьшения ее размера

In [ ]:
# Устанавливаем временный каталог на тот же диск
os.environ['TEMP'] = r'С:\temp'
os.makedirs(os.environ['TEMP'], exist_ok=True)

try:
    # Проверка доступности диска
    test_file = r'С:\test_sqlite_write.tmp'
    with open(test_file, 'wb') as f:
        f.write(b'test')
    os.remove(test_file)
    
    print("✅ Проверка диска успешна")
    
    # Подключаемся к базе с явным указанием timeout
    conn = sqlite3.connect(db_path, timeout=30)
    cursor = conn.cursor()
    
    print("🧹 Оптимизация базы данных...")
    
    # Настройки оптимизации
    cursor.execute("PRAGMA journal_mode = DELETE")
    cursor.execute("PRAGMA page_size = 4096")
    cursor.execute("PRAGMA cache_size = -2000")
    
    # Безопасное сжатие
    print("🔍 Выполняем сжатие...")
    temp_db = r'D:\temp_compressed.db'
    with tqdm(desc="Сжатие", unit="блок") as pbar:
        cursor.execute(f"VACUUM INTO '{temp_db}'")
        pbar.update(1)
    
    conn.close()
    
    # Заменяем оригинальную базу
    os.remove(db_path)
    os.rename(temp_db, db_path)
    
    print("\n✅ База успешно сжата")
    
    # Проверяем новый размер
    size_mb = os.path.getsize(db_path) / (1024 * 1024)
    print(f"📏 Новый размер: {size_mb:.2f} MB")

except Exception as e:
    print(f"\n❌ Ошибка: {str(e)}")
    if 'temp_db' in locals() and os.path.exists(temp_db):
        os.remove(temp_db)

finally:
    if 'conn' in locals():
        conn.close()
    print("\n🔌 Соединение закрыто")

#### Выполним полную оптимизацию и реструктуризацию базы данных SQLite, преобразуя её в высокопроизводительную реляционную базу данных с внешними ключами и улучшенной структурой

In [ ]:
def setup_database_relations(db_path):
    print("⚙️ Начинаем настройку базы данных...")
    
    try:
        # 1. Инициализация подключения
        print("\n🔌 Устанавливаем соединение с базой данных...")
        with tqdm(total=1, desc="Подключение к БД") as pbar:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            pbar.update(1)

        # 2. Применение оптимизированных настроек
        print("\n⚡ Применяем оптимизированные настройки для 64GB RAM...")
        with tqdm(total=8, desc="Настройка параметров") as pbar:
            cursor.executescript("""
            PRAGMA journal_mode = MEMORY;
            PRAGMA synchronous = OFF;
            PRAGMA foreign_keys = ON;
            PRAGMA cache_size = -32000000;  -- 32GB кэша
            PRAGMA mmap_size = 1610612736;  -- 1.5GB mmap
            PRAGMA temp_store = MEMORY;
            PRAGMA locking_mode = EXCLUSIVE;
            PRAGMA threads = 8;
            """)
            pbar.update(8)

        # 3. Анализ структуры базы данных
        print("\n🔍 Анализируем структуру базы данных...")
        with tqdm(total=3, desc="Анализ структуры") as pbar:
            tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
            print("\n📋 Существующие таблицы:", tables['name'].tolist())
            pbar.update(1)
            
            # Проверяем размеры таблиц
            table_sizes = {}
            for table in tables['name']:
                cursor.execute(f"SELECT COUNT(*) FROM {table}")
                table_sizes[table] = cursor.fetchone()[0]
            print("\n📊 Размеры таблиц:", table_sizes)
            pbar.update(1)
            
            # Проверяем текущие индексы
            indexes = pd.read_sql("SELECT * FROM sqlite_master WHERE type='index'", conn)
            print("\n🔖 Существующие индексы:", indexes['name'].tolist())
            pbar.update(1)

        # 4. Создание временных таблиц с улучшенной структурой
        print("\n🔄 Создаем оптимизированные таблицы...")
        with tqdm(total=2, desc="Создание таблиц") as pbar:
            # Таблица weather_new
            cursor.execute("""
            CREATE TABLE IF NOT EXISTS weather_new (
                id INTEGER NOT NULL,
                date TEXT NOT NULL,
                city TEXT NOT NULL,
                latitude REAL,
                longitude REAL,
                temperature_2m REAL,
                relative_humidity_2m REAL,
                rain REAL,
                snowfall REAL,
                snow_depth REAL,
                is_day INTEGER,
                precipitation REAL,
                wind_direction_100m REAL,
                wind_speed_100m REAL,
                FOREIGN KEY (id) REFERENCES cities(id) ON DELETE CASCADE,
                PRIMARY KEY (id, date)
            ) WITHOUT ROWID
            """)
            pbar.update(1)
            
            # Таблица progress_new
            cursor.execute("""
            CREATE TABLE IF NOT EXISTS progress_new (
                id INTEGER NOT NULL,
                city TEXT NOT NULL,
                latitude REAL,
                longitude REAL,
                year INTEGER NOT NULL,
                status TEXT NOT NULL,
                attempts INTEGER DEFAULT 0,
                last_attempt TEXT,
                FOREIGN KEY (id) REFERENCES cities(id) ON DELETE CASCADE,
                PRIMARY KEY (id, year)
            ) WITHOUT ROWID
            """)
            pbar.update(1)

        # 5. Перенос данных с расширенным прогресс-баром
        print("\n📦 Начинаем перенос данных...")
        
        # Временное отключение журналирования для скорости
        cursor.execute("PRAGMA synchronous = OFF")
        cursor.execute("PRAGMA journal_mode = OFF")

        # Функция для переноса данных с прогресс-баром
        def transfer_data(source_table, target_table, batch_size=1000000):
            cursor.execute(f"SELECT COUNT(*) FROM {source_table}")
            total_rows = cursor.fetchone()[0]
            
            with tqdm(total=total_rows, desc=f"Перенос {source_table}", unit="row") as pbar:
                for offset in range(0, total_rows, batch_size):
                    start_time = time.time()
                    
                    # Перенос блока данных
                    cursor.execute(f"""
                    INSERT INTO {target_table}
                    SELECT * FROM {source_table}
                    LIMIT {batch_size} OFFSET {offset}
                    """)
                    
                    # Обновление прогресс-бара
                    rows_processed = min(batch_size, total_rows - offset)
                    pbar.update(rows_processed)
                    
                    # Расчет скорости и оставшегося времени
                    elapsed = time.time() - start_time
                    if elapsed > 0 and offset > 0:
                        speed = rows_processed / elapsed
                        remaining = (total_rows - offset - rows_processed) / speed
                        pbar.set_postfix({
                            "speed": f"{speed:.2f} rows/sec",
                            "remaining": f"{remaining/60:.1f} min"
                        })
                    
                    # Периодический коммит
                    if offset % (10 * batch_size) == 0:
                        conn.commit()

        # Перенос данных для каждой таблицы
        transfer_data("weather", "weather_new")
        transfer_data("progress", "progress_new")

        # 6. Замена оригинальных таблиц
        print("\n🔄 Заменяем оригинальные таблицы...")
        with tqdm(total=5, desc="Замена таблиц") as pbar:
            cursor.execute("PRAGMA locking_mode = EXCLUSIVE")
            pbar.update(1)
            
            cursor.execute("DROP TABLE IF EXISTS weather")
            pbar.update(1)
            
            cursor.execute("ALTER TABLE weather_new RENAME TO weather")
            pbar.update(1)
            
            cursor.execute("DROP TABLE IF EXISTS progress")
            pbar.update(1)
            
            cursor.execute("ALTER TABLE progress_new RENAME TO progress")
            pbar.update(1)

        # 7. Создание индексов
        print("\n📊 Создаем индексы для оптимизации...")
        with tqdm(total=7, desc="Создание индексов") as pbar:
            cursor.execute("PRAGMA synchronous = OFF")
            cursor.execute("PRAGMA journal_mode = MEMORY")
            
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_weather_id ON weather(id)")
            pbar.update(1)
            
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_weather_date ON weather(date)")
            pbar.update(1)
            
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_weather_coords ON weather(latitude, longitude)")
            pbar.update(1)
            
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_progress_id ON progress(id)")
            pbar.update(1)
            
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_progress_year ON progress(year)")
            pbar.update(1)
            
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_progress_coords ON progress(latitude, longitude)")
            pbar.update(1)
            
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_cities_coords ON cities(latitude, longitude)")
            pbar.update(1)

        # 8. Проверка целостности данных
        print("\n🔍 Проверяем целостность данных...")
        with tqdm(total=4, desc="Проверка целостности") as pbar:
            # Быстрая проверка целостности
            cursor.execute("PRAGMA quick_check")
            integrity_check = cursor.fetchone()[0]
            print(f"\n🔎 Результат быстрой проверки: {integrity_check}")
            pbar.update(1)
            
            # Проверка отсутствующих связей
            integrity_checks = {
                "Города без данных в weather": """
                SELECT c.id, c.city FROM cities c
                LEFT JOIN weather w ON c.id = w.id
                WHERE w.id IS NULL
                """,
                
                "Погодные данные с несуществующими городами": """
                SELECT DISTINCT w.id, w.city FROM weather w
                LEFT JOIN cities c ON w.id = c.id
                WHERE c.id IS NULL
                """,
                
                "Прогресс с несуществующими городами": """
                SELECT DISTINCT p.id, p.city FROM progress p
                LEFT JOIN cities c ON p.id = c.id
                WHERE c.id IS NULL
                """
            }
            
            for check_name, query in integrity_checks.items():
                result = pd.read_sql(query, conn)
                if not result.empty:
                    print(f"\n⚠️ {check_name}:")
                    print(result)
                else:
                    print(f"✓ {check_name} - проблем не обнаружено")
                pbar.update(1)

        # 9. Финализация изменений
        print("\n🧹 Оптимизируем базу данных...")
        with tqdm(total=2, desc="Финализация") as pbar:
            cursor.execute("VACUUM")
            pbar.update(1)
            
            # Восстановление безопасного режима
            cursor.executescript("""
            PRAGMA journal_mode = WAL;
            PRAGMA synchronous = NORMAL;
            PRAGMA locking_mode = NORMAL;
            """)
            pbar.update(1)

        print("\n✅ Все изменения успешно применены с максимальной оптимизацией!")
        print("🔒 Не забудьте создать резервную копию базы данных")

    except Exception as e:
        print(f"\n❌ Критическая ошибка: {str(e)}")
        if 'conn' in locals():
            conn.rollback()
            print("Выполнен полный откат изменений")
        traceback.print_exc()
    finally:
        if 'conn' in locals():
            conn.close()
            print("🔌 Соединение с базой данных закрыто")

# Запуск процесса
if __name__ == "__main__":
    db_path = r'C:\Users\Равиль\Проекты для портфолио\9. Проект скрипта для сбора погоды\Скрипт для сбора погоды\all_russian_cities.db'
    
    print("="*50)
    print("🚀 ЗАПУСК ПРОЦЕССА ОПТИМИЗАЦИИ БАЗЫ ДАННЫХ")
    print("="*50)
    
    start_time = time.time()
    setup_database_relations(db_path)
    
    total_time = time.time() - start_time
    print(f"\n🕒 Общее время выполнения: {total_time/60:.2f} минут")

### Создадим скрипты миграции базы данных из SQLite в PostgreSQL

#### Содадим SQL дамп для последующей потоковой загрузки в PostgreSQL для выполнения в командной строке WSL Ubuntu

In [ ]:
sqlite3 all_russian_cities.db .dump | pv -b > dump.sql

#### Выполним преобразование SQLite дампа в PostgreSQL совместимый формат в командной строке WSL Ubuntu

In [ ]:
pv -N "Обработка дампа" dump.sql | sed -E '
  # Удаление SQLite-специфичных команд
  s/PRAGMA foreign_keys=OFF;//g;
  s/\s*WITHOUT ROWID\s*//g;
  /sqlite_sequence/d;
  
  # Преобразование типов данных
  s/AUTOINCREMENT//g;
  s/INTEGER PRIMARY KEY/SERIAL PRIMARY KEY/g;
  s/DATETIME/TIMESTAMP/g;
  
  # Удаление кавычек в именах объектов
  s/CREATE TABLE "([^"]+)"/CREATE TABLE \1/g;
  s/INSERT INTO "([^"]+)"/INSERT INTO \1/g;
  
  # Преобразование BOOLEAN → INTEGER для processed
  s/processed BOOLEAN DEFAULT FALSE/processed INTEGER DEFAULT 0/g;
  s/processed BOOLEAN DEFAULT TRUE/processed INTEGER DEFAULT 1/g;
  s/processed BOOLEAN/processed INTEGER/g;
  
  # Преобразование BOOLEAN → INTEGER для is_day
  s/is_day BOOLEAN/is_day INTEGER/g;
  
  # Замена значений TRUE/FALSE → 1/0 во всех данных
  s/,TRUE,/,1,/g;
  s/,FALSE,/,0,/g;
  s/,TRUE\)/,1)/g;
  s/,FALSE\)/,0)/g;
  s/,TRUE;/,1;/g;
  s/,FALSE;/,0;/g;
' > dump_fixed.sql

#### Создадим базу данных all_russian_cities в WSL Ubuntu

In [ ]:
sudo -u postgres psql -c "CREATE DATABASE all_russian_cities;"

#### Выполним потоковую загрузку SQL-дампа для загрузки базы данных all_russian_cities в PostgreSQL с визуальным мониторингом прогресса и изолированным логированием ошибок в командной строке WSL Ubuntu

In [ ]:
sudo -u postgres bash -c "pv -N 'Загрузка в PostgreSQL' -W -b -t -e -r -a dump_fixed.sql | psql -d all_russian_cities -q 2> /var/lib/postgresql/errors.log"

### Ключевые настройки в Ubuntu WSL и Windows для удаленного доступа к базе данных PostgreSQL из внешней сети

#### Настроим файл /etc/postgresql/16/main/pg_hba.conf с правами sudo в командной строке WSL Ubuntu

In [ ]:
# Database administrative login by Unix domain socket
local   all             postgres                                peer

# TYPE  DATABASE        USER            ADDRESS                 METHOD
# Local connections
local   all             all                                     peer
host    all             all             127.0.0.1/32            scram-sha-256
host    all             all             ::1/128                 scram-sha-256

# Внешний доступ для Datalens:
host    all_russian_cities all          0.0.0.0/0               scram-sha-256

# Локальная сеть и WSL
host    all             all             192.168.1.0/24          scram-sha-256
host    all             all             172.23.64.0/20          scram-sha-256

#### Настроим файл /etc/postgresql/16/main/postgresql.conf с правами sudo в командной строке WSL Ubuntu

In [ ]:
# - Connection Settings -
listen_addresses = '*'          # Прослушивание на всех интерфейсах
port = 5432                     # Стандартный порт PostgreSQL
ssl = on                        # Включение SSL шифрования
ssl_cert_file = '/etc/ssl/certs/ssl-cert-snakeoil.pem'
ssl_key_file = '/etc/ssl/private/ssl-cert-snakeoil.key'

#### Настроим проброс портов между Windows и WSL Ubuntu в Power Shell

netsh interface portproxy add v4tov4 listenaddress=192.168.1.71 listenport=5432 connectaddress=172.23.76.181 connectport=5432

#### Настроим проброс портов в роутере

- Описание: [PostgreSQL Remote Access]
- Интерфейс: [ISP]
- Протокол: [TCP]
- Открыть: [порт] [5432]
- Перенаправить на адрес: [192.168.1.71]
- Новый номер порта назначения: [5432]

#### Настроим сетевой экран антивируса для защиты базы данных от кибератак, чтобы подключаться к базе данных можно было только из локальной сети и из внешней сети только с доверенных IP адресов <a href='https://yandex.cloud/ru/docs/troubleshooting/datalens/how-to/ip-address-range'>Yandex DataLens</a>.

##### Настроим пакетное правило "PostgreSQL Allow Datalens 1"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [178.154.242.128/28]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 2"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [178.154.242.144/28]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 3"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [178.154.242.160/28]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 4"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [178.154.242.176/28]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 5"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [178.154.242.192/28]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 6"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [178.154.242.208/28]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 7"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [130.193.60.0/28]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 8"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [2a02:6b8:c03:500:0:f83d:a987:0/112]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 9"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [2a02:6b8:c02:900:0:f83d:a987:0/112]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 10"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [2a02:6b8:c0e:500:0:f83d:a987:0/112]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Allow Datalens 11"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [2a02:6b8:c41:1300:0:f83d:a987:0/112]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL Домашняя сеть"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [192.168.1.0/24]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL WSL"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [172.23.64.1/32]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL WSL 2"

- Действие: [Разрешать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [адреса из списка] [удаленные адеса] [172.23.76.181/32]
- Записывать события: [✔]

##### Настроим пакетное правило "PostgreSQL БЛОКИРОВАТЬ ВСЁ"

- Действие: [Запрещать]
- Направление: [входящее]
- Протокол: [TCP]
- Локальные порты: [5432]
- Тип адреса: [любой адрес]
- Записывать события: [✔]

### Создадим пользователя datalens_user с ограниченными правами на чтение таблиц базы данных all_russian_cities в WSL Ubuntu

In [ ]:
sudo -u postgres psql -d all_russian_cities -c "
CREATE USER datalens_user WITH PASSWORD '{PASSWORD}';
GRANT CONNECT ON DATABASE all_russian_cities TO datalens_user;
GRANT USAGE ON SCHEMA public TO datalens_user;
GRANT SELECT ON TABLE cities, progress, weather TO datalens_user;
"

### Тестирование подключения к базе данных

#### Устанавливаем параметры

In [6]:
DB_USER = "datalens_user"
DB_PASSWORD = '{PASSWORD}'
DB_HOST = "94.41.20.20"
DB_PORT = "5432"
DB_NAME = "all_russian_cities"

#### Кодируем пароль для URL

In [7]:
encoded_password = quote_plus(DB_PASSWORD)

#### Формируем строку подключения

In [8]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

#### Сохраняем коннектор

In [9]:
engine = create_engine(connection_string)

#### Проверяем подключение к базе данных all_russian_cities

In [10]:
try:
    print("Пытаемся подключиться к PostgreSQL...")
    tables = pd.read_sql("SELECT table_name FROM information_schema.tables WHERE table_schema='public'", engine)
    print("\nУспешное подключение к PostgreSQL!")
    result = engine.connect().execute(text("SELECT version()"))
    print(f"\nВерсия PostgreSQL: {result.scalar()}")
    print("\nДоступные таблицы в базе данных:")
    display(tables)
except Exception as e:
    print(f"Ошибка: {e}")

Пытаемся подключиться к PostgreSQL...

Успешное подключение к PostgreSQL!

Версия PostgreSQL: PostgreSQL 16.14 (Ubuntu 16.14-0ubuntu0.24.04.1) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit

Доступные таблицы в базе данных:


,table_name
0,cities
1,weather
2,progress


#### Выполняем тестовый запрос

In [11]:
query = """
        SELECT *
        FROM weather
        ORDER by id
        LIMIT 10
        """
df = pd.read_sql(query, engine)
display(df.T)

,0,1,2,3,4,5,6,7,8,9
id,1,1,1,1,1,1,1,1,1,1
date,1975-01-01 05:00:00,1975-01-01 11:00:00,1975-01-01 17:00:00,1975-01-01 23:00:00,1975-01-02 05:00:00,1975-01-02 11:00:00,1975-01-02 17:00:00,1975-01-02 23:00:00,1975-01-03 05:00:00,1975-01-03 11:00:00
city,Абаза,Абаза,Абаза,Абаза,Абаза,Абаза,Абаза,Абаза,Абаза,Абаза
latitude,52.651054,52.651054,52.651054,52.651054,52.651054,52.651054,52.651054,52.651054,52.651054,52.651054
longitude,90.10116,90.10116,90.10116,90.10116,90.10116,90.10116,90.10116,90.10116,90.10116,90.10116
temperature_2m,-17.1815,-13.9815,-14.6315,-17.6815,-16.2315,-16.1315,-18.2315,-16.2315,-11.7815,-9.8815
relative_humidity_2m,74.51296,78.67868,80.24075,81.14631,67.63998,72.50379,78.29689,74.37779,66.68781,69.37403
rain,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
snowfall,0.0,0.42,0.42,0.28,0.0,0.0,0.0,0.0,0.0,0.0
snow_depth,0.53,0.54,0.55,0.55,0.55,0.55,0.55,0.55,0.55,0.55


### Настроим автоматический запуск PostgreSQL сервера при запуске ПК

#### Создадим файл wsl_start.bat со следующим содержимым и разместим его в папке автозапуска Windows
C:\Users\Равиль\AppData\Roaming\Microsoft\Windows\Start Menu\Programs\Startup\

In [ ]:
@echo off
chcp 65001 >nul
echo Starting WSL and PostgreSQL...

wsl -d Ubuntu bash -c "sudo systemctl start postgresql; echo 'PostgreSQL started'; sleep infinity"

echo WSL and PostgreSQL are running in background
echo Keep this window open!
pause

#### Отредактируем файл visudo с правами sudo в WSL Ubuntu и добавим следующее содержимое для того, чтобы не требовался ввод пароля при запуске PostgreSQL сервера

In [ ]:
ravil ALL=(ALL) NOPASSWD: /usr/bin/systemctl start postgresql

## Визуализация и анализ

Визуализация результатов проекта выполнена в Yandex DataLens Public. Дашборд можно посмотреть по <a href='https://datalens.yandex/fe071bz8rmjg0?tab=pLM'>ссылке</a>.